<a href="https://colab.research.google.com/github/ondrekm/stanford-cars-model-comparison/blob/main/Elemz%C3%A9s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ANALYSIS_PLOTS

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class CFG:
    save_root: str = "/content/drive/MyDrive/stanford_cars_project/data"
    output_root: str = "/content/drive/MyDrive/stanford_cars_project/outputs"

cfg = CFG()

os.makedirs(cfg.output_root, exist_ok=True)

print("SAVE ROOT:", cfg.save_root)
print("OUTPUT ROOT:", cfg.output_root)

In [ ]:
required_files = [
    os.path.join(cfg.save_root, "multi_run_results_10.csv"),
    os.path.join(cfg.save_root, "multi_run_histories_10.csv"),
    os.path.join(cfg.output_root, "stanford_cars_experiment_summary.csv"),
    os.path.join(cfg.output_root, "stanford_cars_experiment_clustering.csv"),
    os.path.join(cfg.output_root, "stanford_cars_experiment_retrieval.csv"),
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    raise FileNotFoundError(f"Hiányzó fájl(ok): {missing}")

print("Minden szükséges fájl elérhető.")

In [ ]:
multi_run_results_df = pd.read_csv(os.path.join(cfg.save_root, "multi_run_results_10.csv"))
multi_run_histories_df = pd.read_csv(os.path.join(cfg.save_root, "multi_run_histories_10.csv"))
summary_df_raw = pd.read_csv(os.path.join(cfg.output_root, "stanford_cars_experiment_summary.csv"))
clustering_df = pd.read_csv(os.path.join(cfg.output_root, "stanford_cars_experiment_clustering.csv"))
retrieval_df = pd.read_csv(os.path.join(cfg.output_root, "stanford_cars_experiment_retrieval.csv"))

print("multi_run_results_df shape:", multi_run_results_df.shape)
print("multi_run_histories_df shape:", multi_run_histories_df.shape)
print("summary_df_raw shape:", summary_df_raw.shape)
print("clustering_df shape:", clustering_df.shape)
print("retrieval_df shape:", retrieval_df.shape)

In [ ]:
summary_df = summary_df_raw.copy()

# Ha az első oszlop neve pl. "Unnamed: 0", akkor az a model_name
if summary_df.columns[0].startswith("Unnamed"):
    summary_df = summary_df.rename(columns={summary_df.columns[0]: "model_name"})

# Oszlopnevek egységesítése
clean_cols = []
for c in summary_df.columns:
    c2 = str(c).replace("(", "").replace(")", "").replace("'", "").replace('"', "")
    c2 = c2.replace(",", "_").replace(" ", "")
    c2 = c2.replace("__", "_")
    clean_cols.append(c2)

summary_df.columns = clean_cols

display(summary_df.head())
print(summary_df.columns.tolist())

In [ ]:
import os
import pandas as pd

summary_path = os.path.join(cfg.output_root, "stanford_cars_experiment_summary.csv")

# Két fejlécsorral olvassuk be
summary_df = pd.read_csv(summary_path, header=[0, 1])

# Oszlopnevek kilapítása
flat_cols = []
for col in summary_df.columns:
    left, right = col
    left = str(left).strip()
    right = str(right).strip()

    if left.startswith("Unnamed"):
        flat_cols.append(right)
    elif right.startswith("Unnamed") or right == "":
        flat_cols.append(left)
    else:
        flat_cols.append(f"{left}_{right}")

summary_df.columns = flat_cols

# A teljesen üres sorok kidobása
summary_df = summary_df.dropna(how="all").reset_index(drop=True)

print(summary_df.columns.tolist())
display(summary_df)

In [ ]:
import os
import pandas as pd

summary_path = os.path.join(cfg.output_root, "stanford_cars_experiment_summary.csv")

# Két fejlécsorral olvassuk be
summary_df = pd.read_csv(summary_path, header=[0, 1])

# Oszlopnevek kilapítása
flat_cols = []
for col in summary_df.columns:
    left, right = col
    left = str(left).strip()
    right = str(right).strip()

    if left.startswith("Unnamed"):
        flat_cols.append(right)
    elif right.startswith("Unnamed") or right == "":
        flat_cols.append(left)
    else:
        flat_cols.append(f"{left}_{right}")

summary_df.columns = flat_cols

# A teljesen üres sorok kidobása
summary_df = summary_df.dropna(how="all").reset_index(drop=True)

print(summary_df.columns.tolist())
display(summary_df)

In [ ]:
import os
import pandas as pd

summary_path = os.path.join(cfg.output_root, "stanford_cars_experiment_summary.csv")

summary_df = pd.read_csv(summary_path, header=[0, 1])

# MultiIndex oszlopok kilapítása
flat_cols = []
for col in summary_df.columns:
    left, right = col
    left = str(left).strip()
    right = str(right).strip()

    if left.startswith("Unnamed"):
        flat_cols.append(right if right else left)
    elif right.startswith("Unnamed") or right == "":
        flat_cols.append(left)
    else:
        flat_cols.append(f"{left}_{right}")

summary_df.columns = flat_cols

# Az első oszlop átnevezése model_name-ra
summary_df = summary_df.rename(columns={summary_df.columns[0]: "model_name"})

# Az első sor valójában hibás fejlécsor-maradvány, dobjuk ki
summary_df = summary_df[summary_df["model_name"] != "model_name"].copy()

# Index újra
summary_df = summary_df.reset_index(drop=True)

# Számmá alakítás
for col in summary_df.columns:
    if col != "model_name":
        summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

print(summary_df.columns.tolist())
display(summary_df)

In [ ]:
# STABILITY PLOT – futásonkénti szórás

import os
import numpy as np
import matplotlib.pyplot as plt

plot_df = multi_run_results_df.copy()

plt.figure(figsize=(10, 5))

run_order = plot_df.sort_values("test_acc", ascending=False)["run_name"].tolist()
run_order = list(dict.fromkeys(run_order))

for i, run_name in enumerate(run_order):
    subset = plot_df[plot_df["run_name"] == run_name]
    xvals = np.full(len(subset), i)
    plt.scatter(xvals, subset["test_acc"])

plt.xticks(range(len(run_order)), run_order, rotation=60)
plt.ylabel("Test accuracy")
plt.title("Futások stabilitása és szórása")
plt.tight_layout()

out_path = os.path.join(cfg.output_root, "stability_plot.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Mentve:", out_path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

results_path = os.path.join(cfg.save_root, "multi_run_results_10.csv")
df = pd.read_csv(results_path)

def extract_group(run_name: str) -> str:
    if run_name.startswith("baseline_rep_"):
        return "baseline_rep"
    if run_name.startswith("full_aug_rep_"):
        return "full_aug_rep"
    if run_name.startswith("baseline_"):
        return "baseline"
    if run_name.startswith("jit"):
        return "jit02"
    if run_name.startswith("affine"):
        return "affine"
    if run_name.startswith("erasing"):
        return "erasing"
    if run_name.startswith("weighted"):
        return "weighted"
    if run_name.startswith("full_aug"):
        return "full_aug"
    return run_name

df["group_name"] = df["run_name"].apply(extract_group)

group_summary = df.groupby("run_name").agg({
    "test_acc": ["mean", "std"]
})

group_summary.columns = ["test_acc_mean", "test_acc_std"]
group_summary = group_summary.reset_index()

plt.figure(figsize=(10, 5))
plt.bar(
    group_summary["run_name"],
    group_summary["test_acc_mean"],
    yerr=group_summary["test_acc_std"]
)

plt.xticks(rotation=45)
plt.ylabel("Test accuracy")
plt.title("Futtatások összehasonlítása (átlag ± szórás)")
plt.tight_layout()

out_path = os.path.join(cfg.output_root, "run_comparison.png")
plt.savefig(out_path)
plt.show()

print("Mentve:", out_path)

In [ ]:
candidate_runs = [
    "baseline_s42",
    "full_aug",
    "weighted",
    "jit02",
    "baseline_rep_s42",
]

available_runs = [r for r in candidate_runs if r in multi_run_histories_df["run_name"].unique()]

if len(available_runs) == 0:
    available_runs = list(multi_run_histories_df["run_name"].unique())[:3]

plt.figure(figsize=(8, 5))

for run_name in available_runs:
    subset = multi_run_histories_df[multi_run_histories_df["run_name"] == run_name].sort_values("epoch")
    plt.plot(subset["epoch"], subset["val_acc"], label=run_name)

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Tanulási görbék összehasonlítása")
plt.legend()
plt.tight_layout()

out_path = os.path.join(cfg.output_root, "training_curves.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Mentve:", out_path)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(clustering_df["pc1"], clustering_df["pc2"], c=clustering_df["cluster"], s=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Embedding tér klaszterezése (PCA + KMeans)")
plt.tight_layout()

out_path = os.path.join(cfg.output_root, "kmeans_clusters.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Mentve:", out_path)

In [ ]:
print("Retrieval eredmények:")
display(retrieval_df)

In [ ]:
retrieval_row = retrieval_df.iloc[0]
metric_names = retrieval_row.index.tolist()
metric_values = retrieval_row.values.astype(float)

plt.figure(figsize=(6, 4))
plt.bar(metric_names, metric_values)
plt.ylabel("Érték")
plt.title("Retrieval eredmények")
plt.tight_layout()

out_path = os.path.join(cfg.output_root, "retrieval_metrics.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Mentve:", out_path)

In [ ]:
best_runs_df = multi_run_results_df.sort_values("test_acc", ascending=False).reset_index(drop=True)
display(best_runs_df.head(10))

In [ ]:
import os
import pandas as pd

# Ha még nincs meg
best_runs_df = multi_run_results_df.sort_values("test_acc", ascending=False).reset_index(drop=True)

cols_to_show = [
    "run_name", "seed", "model_name", "epochs", "batch_size", "lr",
    "img_size", "train_fraction", "color_jitter",
    "use_affine", "use_erasing", "weighted_sampling",
    "best_val_acc", "test_acc", "time_min"
]

available_cols = [c for c in cols_to_show if c in best_runs_df.columns]
experiment_table_df = best_runs_df[available_cols].copy()

out_path = os.path.join(cfg.output_root, "experiment_config_table.csv")
experiment_table_df.to_csv(out_path, index=False)

print("Mentve:", out_path)
print("Létezik:", os.path.exists(out_path))

display(experiment_table_df.head(10))

In [ ]:
generated_files = [
    "stability_plot.png",
    "training_curves.png",
    "kmeans_clusters.png",
    "retrieval_metrics.png",
    "experiment_config_table.csv",
    "configuration_comparison.png",
]

print("Generált fájlok:")
for fn in generated_files:
    full = os.path.join(cfg.output_root, fn)
    print(full, "->", os.path.exists(full))